In [1]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import os
import numpy as np
import seaborn as sns

In [2]:
username = os.path.expanduser('~')
username

'/home/ehdgus7279'

In [3]:
# 소수점 아래 10자리까지 출력
pd.options.display.float_format = '{:.10f}'.format

In [4]:
benchmarks = [
    # Low-sharing Apps
    "fdtd2d", "jacobi1d", "fastwalshtransform", "relu", "fft", "fir", 
    "sssp", "convolution2d", "color", "bitonicsort", "pagerank", "simpleconvolution",

    # High-sharing Apps
    'gups',    
    'syr2k',
    'mm2', 
    'mm3', 
    'gemm',
    'gaussian',
    'gramschmidt',
    'vgg16',
    'spmv',
    'mis',
    'stencil2d',
    'lu',
    'matrixmultiplication',
    'matrixtranspose',
    'im2col',
    'nbody',
    'bfs',
]

In [5]:
low_sharing_benchmarks = [
    "fdtd2d", "jacobi1d", "fastwalshtransform", "relu", "fft", "fir", 
    "sssp", "convolution2d", "color", "bitonicsort", "pagerank", "simpleconvolution",
]

high_sharing_benchmarks = [
    'gups',    
    'syr2k',
    'mm2', 
    'mm3', 
    'gemm',
    'gaussian',
    'gramschmidt',
    'vgg16',
    'spmv',
    'mis',
    'stencil2d',
    'lu',
    'matrixmultiplication',
    'matrixtranspose',
    'im2col',
    'nbody',
    'bfs',
]

In [6]:
num_chiplets = 36
curr_chip_cols = [str(i) for i in range(36)]

clusters_3x3 = {
    'cluster_0_3x3': [0, 1, 2, 6, 7, 8, 12, 13, 14],
    'cluster_1_3x3': [3, 4, 5, 9, 10, 11, 15, 16, 17],
    'cluster_2_3x3': [18, 19, 20, 24, 25, 26, 30, 31, 32],
    'cluster_3_3x3': [21, 22, 23, 27, 28, 29, 33, 34, 35],
}

clusters_2x2 = {
    'cluster_0_2x2': [0, 1, 6, 7],       # 좌측 상단
    'cluster_1_2x2': [2, 3, 8, 9],       # 상단 중앙
    'cluster_2_2x2': [4, 5, 10, 11],     # 우측 상단
    'cluster_3_2x2': [12, 13, 18, 19],   # 좌측 중앙
    'cluster_4_2x2': [14, 15, 20, 21],   # 정중앙
    'cluster_5_2x2': [16, 17, 22, 23],   # 우측 중앙
    'cluster_6_2x2': [24, 25, 30, 31],   # 좌측 하단
    'cluster_7_2x2': [26, 27, 32, 33],   # 하단 중앙
    'cluster_8_2x2': [28, 29, 34, 35]    # 우측 하단
}

clusters_3x3_cols = [k for k in clusters_3x3.keys()]
clusters_2x2_cols = [k for k in clusters_2x2.keys()]

chiplet_to_cluster_2x2 = {}
for cluster_name, chiplet_ids in clusters_2x2.items():
    for chip_id in chiplet_ids:
        chiplet_to_cluster_2x2[chip_id] = cluster_name

chiplet_to_cluster_3x3 = {}
for cluster_name, chiplet_ids in clusters_3x3.items():
    for chip_id in chiplet_ids:
        chiplet_to_cluster_3x3[chip_id] = cluster_name

In [7]:
def get_cluster_id(id):
    row = id // 6
    col = id % 6

    partRow = row // 3
    partCol = col // 3

    return partRow * 2 + partCol

In [9]:
base_dir=f'{username}/mgvm/results/memory-traces'
date = "20260329-231025-mesh"
expanded_dir = f"{base_dir}/{date}/expanded"
trace_dir = f"{base_dir}/{date}"
print(f"{base_dir}/{date}")

/home/ehdgus7279/mgvm/results/memory-traces/20260329-231025-mesh


In [70]:
benchmark = 'gups'
trace_file = f"{expanded_dir}/{benchmark}_mesh_6x6_memory_trace.csv"

df = pd.read_csv(trace_file, skipinitialspace=True)
df

,time,pc,addr,owner,rw,0,1,2,3,4,...,28,29,30,31,32,33,34,35,access_count,shared_chiplets
0,0.0040940600,100663552,2248937472,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
1,0.0040941120,100663552,2248937472,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
2,0.0040941150,100663552,2248937472,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
3,0.0040941260,100663552,2248937472,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
4,0.0040941490,100663552,2248937472,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32193269,0.0244230210,100663692,2197413824,5,1,0,0,0,0,0,...,0,0,0,0,0,0,0,1,1,1
32193270,0.0244230220,100663692,2202185728,6,1,0,0,0,0,0,...,0,0,0,0,0,0,0,1,1,1
32193271,0.0244230220,100663792,2220765184,2,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
32193272,0.0244230230,100663792,2178607104,6,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1


In [71]:
rdma_df = df.copy()

for i in range(num_chiplets):
    col = str(i)
    rdma_df[col] = np.where(rdma_df['owner'] == i, 0, rdma_df[col])

df['is_rdma'] = rdma_df[curr_chip_cols].any(axis=1)

In [74]:
rdma_df = df[df['is_rdma'] == True]
rdma_df

,time,pc,addr,owner,rw,0,1,2,3,4,...,access_count,shared_chiplets,is_rdma,cluster_0_2x2,cluster_1_2x2,cluster_2_2x2,cluster_3_2x2,cluster_4_2x2,cluster_5_2x2,cluster_6_2x2
0,0.0040940600,100663552,2248937472,0,0,0,0,0,0,0,...,1,1,True,0,0,0,0,0,0,1
1,0.0040941120,100663552,2248937472,0,0,0,0,0,0,0,...,1,1,True,0,0,0,1,0,0,0
2,0.0040941150,100663552,2248937472,0,0,0,0,0,0,0,...,1,1,True,0,0,0,1,0,0,0
3,0.0040941260,100663552,2248937472,0,0,0,0,0,0,0,...,1,1,True,1,0,0,0,0,0,0
5,0.0040945800,100663552,2248937472,0,0,0,0,0,0,0,...,1,1,True,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32193269,0.0244230210,100663692,2197413824,5,1,0,0,0,0,0,...,1,1,True,0,0,0,0,0,0,0
32193270,0.0244230220,100663692,2202185728,6,1,0,0,0,0,0,...,1,1,True,0,0,0,0,0,0,0
32193271,0.0244230220,100663792,2220765184,2,0,0,0,0,0,0,...,1,1,True,0,0,0,0,0,0,0
32193272,0.0244230230,100663792,2178607104,6,0,0,0,0,0,0,...,1,1,True,0,0,0,1,0,0,0


In [75]:
import math
import mmh3
from collections import OrderedDict

In [77]:
class HLLUnit:
    def __init__(self, b=6):
        self.b = b
        self.m = 1 << b
        self.registers = np.zeros(self.m, dtype=np.uint8)
        self.access_count = 0

    def add(self, addr):
        self.access_count += 1
        # 1. 해시값을 32비트 무부호 정수로 고정 (중요!)
        # signed=False로 하여 음수 방지
        x = mmh3.hash(str(addr), signed=False) & 0xFFFFFFFF
        
        # 2. 버킷 인덱스 추출 (하위 b비트)
        idx = x & (self.m - 1)
        
        # 3. 연속된 0의 개수(rho) 계산 (나머지 상위 비트 사용)
        # x를 b만큼 밀어서 인덱스 비트를 제거
        w = x >> self.b
        
        if w > 0:
            # (남은 비트 전체 길이) - (1이 나타나는 위치) + 1
            # 예: w가 0...0100 이면 bit_length는 3, rho는 (32-6)-3+1 = 24
            rho = (32 - self.b) - w.bit_length() + 1
        else:
            # w가 0이라는 건 상위 비트가 전부 0이라는 뜻 (매우 희귀한 케이스)
            rho = (32 - self.b) + 1
            
        # 4. 기존 값과 비교하여 최대값 갱신
        if rho > self.registers[idx]:
            self.registers[idx] = rho

    def estimate(self):
        m = self.m
        # 빈 버킷 개수 확인 (Linear Counting용)
        v = np.count_nonzero(self.registers == 0)
        
        if v == m: return 0

        # 기본 HLL 추정 공식
        alpha = 0.7213 / (1 + 1.079 / m)
        # 조화 평균 계산: sum(2^-register)
        term_sum = np.sum(1.0 / (2.0 ** self.registers))
        est = alpha * (m**2) / term_sum

        # --- 보정 로직 (이게 핵심) ---
        # 1. 데이터가 적을 때 (Small Range)
        if est <= 2.5 * m:
            if v > 0:
                est = m * math.log(m / v)
        
        # 2. 데이터가 너무 많을 때 (Large Range - 32비트 해시 한계 근접 시)
        # 32비트 해시 기준이므로 2^32 근처에서 보정 필요 (보통 150만 개면 해당 안 됨)
        if est > (1/30) * (2**32):
            est = -(2**32) * math.log(1 - est/(2**32))

        return max(1, int(round(est)))

In [78]:
def simulate_realtime_hll(df, num_entries=1000, bits=4):
    # 하드웨어 리소스: 10개의 HLL 엔트리를 관리하는 LRU 캐시 구조
    hll_table = OrderedDict() 
    
    # 결과 비교를 위해 실제 정답(Ground Truth)을 담을 딕셔너리
    actual_unique_set = {} # {pc: set(addrs)}
    actual_total_access = {} # {pc: count}

    print(f"Starting Real-time Simulation (Entries: {num_entries})...")

    pcs = df['pc'].values
    addrs = df['addr'].values

    for i in range(len(pcs)):
        pc = pcs[i]
        addr = addrs[i]

        # 1. 실제 정답 업데이트 (비교용)
        if pc not in actual_unique_set:
            actual_unique_set[pc] = set()
            actual_total_access[pc] = 0
        actual_unique_set[pc].add(addr)
        actual_total_access[pc] += 1

        # 2. 하드웨어 HLL 유닛 동작
        if pc in hll_table:
            # 기존에 추적 중인 PC면 위치만 최신으로 갱신(LRU) 후 업데이트
            hll_table.move_to_end(pc)
            hll_table[pc].add(addr)
        else:
            # 새로운 PC가 들어왔는데 자리가 없다면 가장 오래된 놈 방출
            if len(hll_table) >= num_entries:
                hll_table.popitem(last=False) 
            
            # 새 HLL 엔트리 할당
            hll_table[pc] = HLLUnit(b=bits)
            hll_table[pc].add(addr)

    # 3. 시뮬레이션 종료 후 결과 비교
    # 현재 하드웨어 테이블에 살아남은 'Top 10' PC들에 대해 검증
    results = []

    for pc, unit in hll_table.items():
        est_unique = unit.estimate()
        real_unique = len(actual_unique_set[pc])
        
        # 0으로 나누는 것을 방지 (최소 1로 취급하거나 0으로 결과 처리)
        hll_reuse = round(unit.access_count / est_unique, 2) if est_unique > 0 else 0
        real_reuse = round(actual_total_access[pc] / real_unique, 2) if real_unique > 0 else 0
        
        # Error Rate 계산 시에도 분모 체크
        error_rate = round(abs(real_unique - est_unique) / real_unique * 100, 2) if real_unique > 0 else 0
        
        results.append({
            'PC': hex(int(pc)),
            'Actual_Access': actual_total_access[pc],
            'Actual_Unique': real_unique,
            'HLL_Estimate': est_unique,
            'HLL_Avg_Reuse': hll_reuse,
            'Real_Avg_Reuse': real_reuse,
            'Error_Rate(%)': error_rate
        })

    return pd.DataFrame(results).sort_values(by='Actual_Access', ascending=False)

In [83]:
final_res = simulate_realtime_hll(rdma_df, num_entries=1000, bits=6)

Starting Real-time Simulation (Entries: 1000)...


In [ ]:
final_res

,PC,Actual_Access,Actual_Unique,HLL_Estimate,HLL_Avg_Reuse,Real_Avg_Reuse,Error_Rate(%)
12,0x60001f0,16559893,524281,497566,33.2800000000,31.5900000000,5.1000000000
11,0x600018c,14798483,524278,497566,29.7400000000,28.2300000000,5.1000000000
3,0x6000184,3968,3968,4345,0.9100000000,1.0000000000,9.5000000000
4,0x6000188,3960,3960,4408,0.9000000000,1.0000000000,11.3100000000
10,0x60001dc,2214,1965,2065,1.0700000000,1.1300000000,5.0900000000
9,0x60001d8,743,662,634,1.1700000000,1.1200000000,4.2300000000
7,0x60001e4,127,127,119,1.0700000000,1.0000000000,6.3000000000
6,0x60001e0,110,110,126,0.8700000000,1.0000000000,14.5500000000
8,0x60001e8,77,77,93,0.8300000000,1.0000000000,20.7800000000
5,0x60001ec,58,58,65,0.8900000000,1.0000000000,12.0700000000


: 